# ReportSPPR v5.1

Сборка от 02.10.2026. Новый формат отчёта — 11 слайдов, как макет v2.5 (`examples/mockup_report_v2.html`). Цифры и вёрстку делает модуль `report_v5.py` (в репозитории — `src/report_v5.py`), тот же, что собирает макет; оболочка страницы — `template_v5.html`. Имена файлов — параметры `MODULE_FILE` и `TEMPLATE_FILE` в ячейке 1; мажор модуля и шаблона должен совпадать с мажором ноутбука, иначе ячейка 1 остановится до вызовов ИИ.

**Как запускать.** Секрет `DEEPSEEK_API_KEY` (или `GEMINI_API_KEY` при `AI_PROVIDER = "google"`) — в секретах Colab. В окне загрузки выбрать выгрузку `sppr_dump_*.json` и, если их нет на Диске в `REPORTS/SPPR`, — `template_v5.html` и `report_v5.py`; при работе без Диска — ещё `history.json` и `markup_cache_v5.json` с прошлого запуска. Затем «Среда выполнения → Выполнить всё». Прогон — до часа: качество важнее скорости. Этапы, время и токены печатаются по ходу и попадают в паспорт данных отчёта.

**Что нового в 5.1** (минор, шаблон не менялся): блок «Контроль решений прошлой недели» скрыт до слайда поручений руководителя Т-70 — по решениям ИИ это контроль идей ИИ, а не поручений (`docs/my-FT.md`, В-21); модель больше не ставит статусы прошлым решениям; решения недели по-прежнему пишутся в историю. Модуль — `report_v5.py` 5.1.

**Что нового в 5.0** (мажор: новый шаблон, новый состав данных отчёта):
- отчёт в формате макета: 11 слайдов, разбор ИИ внизу каждого (Т-43); состав — `HANDOFF.md`, разд. 0, требования — `docs/my-FT.md`;
- разметка решённых по схеме v5 (тип по сути со служебными карточками, причина правки с обоснованием, риск повтора, «мог сделать сам»), группы тем, разметка открытых старше 30 дней, критичных открытых инцидентов, планшетов и скрытых повторов;
- разборы слайдов — JSON «факт → вывод → действие → кому → эффект»: цифры подставляет код по ключам, номера обращений проверяются по выгрузке, ошибка — повторный запрос с перечнем ошибок;
- срок решения — 8 ч в рабочем дне, выходные и праздники не считаются (Т-67, Т-68);
- история недель v2: поток по типам и месяцам, решения руководителя — для контроля в следующем отчёте (Т-52);
- кэш всех ответов ИИ: повторный прогон той же выгрузки — тот же отчёт без оплаты (`REUSE_AI`); кэш разметки по id и «Н/Д» при сбое — из v4.1;
- шаблон и модуль ищутся и проверяются в ячейке 1, до платных вызовов; Приложение 1 (TF-IDF) не перенесено.

Порядок ячеек менять нельзя: они делят общие переменные (`R`, `AI_MARKUP`, `AI_CACHE`, `RUN_LOG`, `HISTORY`, клиенты ИИ).

# БЛОК 1: ПАРАМЕТРЫ, КЛИЕНТЫ ИИ, ФАЙЛЫ, ОТЧЁТНАЯ НЕДЕЛЯ

In [4]:
# =====================================================================
# [ЯЧЕЙКА 1] ПАРАМЕТРЫ, КЛИЕНТЫ ИИ, ФАЙЛЫ, ОТЧЁТНАЯ НЕДЕЛЯ
# =====================================================================
# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО (v5.0): ячейка собрана заново из ячейки 1 v4.2. Расчёт и вёрстка — модуль report_v5.py,
#               оболочка — template_v5.html; оба файла ищутся и проверяются ДО платных вызовов ИИ;
#               выгрузка читается без pandas; неделя задаётся параметром REPORT_WEEK.
# ПОЧЕМУ: my-FT Р-6 — ноутбук сам собирает отчёт в формате макета; один модуль с макетом (В-18 = а);
#         в v4 шаблон искался в самом конце — после часа разметки прогон мог упасть без отчёта
# ----------------------------------------------------------------------
import os, io, re, csv, glob, json, time, math, hashlib, asyncio, importlib.util
from datetime import datetime
from google.colab import files, userdata

NOTEBOOK_VERSION = "5.1"                    # мажор ноутбука = мажор модуля и шаблона
MODULE_FILE = 'report_v5.py'                # расчёт и вёрстка (в репозитории — src/report_v5.py)
TEMPLATE_FILE = 'template_v5.html'          # оболочка отчёта (в репозитории — template_v5.html)

# --- ПРОВАЙДЕР И МОДЕЛИ ---
AI_PROVIDER = "deepseek" #@param ["google", "deepseek"]
GEMINI_BATCH_MODEL = 'gemini-3.6-flash'
GEMINI_SUMMARY_MODEL = 'gemini-3.1-pro'
DEEPSEEK_BATCH_MODEL = 'deepseek-chat'      # разметка (HANDOFF, разд. 0: качество важнее скорости)
DEEPSEEK_SUMMARY_MODEL = 'deepseek-reasoner'  # разборы слайдов — модель с рассуждением
BATCH_MODEL = GEMINI_BATCH_MODEL if AI_PROVIDER == "google" else DEEPSEEK_BATCH_MODEL
SUMMARY_MODEL = GEMINI_SUMMARY_MODEL if AI_PROVIDER == "google" else DEEPSEEK_SUMMARY_MODEL

BATCH_SIZE = 15                 # стартовый размер пачки разметки; при обрыве ответа — пополам
MAX_CONCURRENCY = 4             # одновременных вызовов модели
SLEEP_TIME = 2                  # пауза перед повтором, удваивается; ждёт вне слота
MAX_OUTPUT_TOKENS = 8192        # потолок ответа модели разметки на пачку
PROMPT_TEXT_LIMIT = 1500        # обрезка описания и решения в промпте разметки
API_TIMEOUT = 900               # сек: DeepSeek под нагрузкой держит соединение до 10 минут (прогон 01.10 — пачки по 600 с)

# --- ПРОГОН ---
REPORT_WEEK = 0 #@param {type:"integer"}
# ЧТО ИЗМЕНЕНО (v5.0): REPORT_WEEK — номер ISO-недели отчёта; 0 — неделя максимальной даты решения (Т-12)
REUSE_AI = True #@param {type:"boolean"}
# ЧТО ИЗМЕНЕНО (v5.0): повторный прогон той же выгрузки берёт ответы ИИ из кэша — тот же отчёт без оплаты (Т-63, Т-65)
TEST_SAMPLE_SIZE = 0            # >0 — размечаются первые N решённых и N старых открытых, история не пишется

# --- ДИСК, ФАЙЛЫ ---
DRIVE_MODE = "always" #@param ["auto", "always", "never"]
USE_FILE_PICKER = True #@param {type:"boolean"}
CACHE_DIR = '/content/sppr_cache'                          # кэш сессии: шаблон, модуль, история, кэш ИИ
DRIVE_ROOT = '/content/drive/MyDrive/PROJECTs/ReportSPPRv1'         # шаблон и модуль на Диске ищутся здесь
HISTORY_DIR = '/content/drive/MyDrive/PROJECTs/ReportSPPRv1/result/history'
if USE_FILE_PICKER:
    TARGET_DIR = '/content/result'
else:
    SOURCE_DIR_VARIANTS = ['/content/drive/MyDrive/PROJECTs/ReportSPPRv1/source', '/content/drive/MyDrive/reports/sppr/source']
    TARGET_DIR = '/content/drive/MyDrive/PROJECTs/ReportSPPRv1/result'
AI_CACHE_FILE = os.path.join(CACHE_DIR, 'markup_cache_v5.json')

# --- КЛИЕНТЫ ИИ ---
# Т-66: ключ DeepSeek нужен только при AI_PROVIDER = "deepseek"; ключи — только из секретов Colab
client_gemini = client_deepseek_async = None
if AI_PROVIDER == "google":
    from google import genai
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    if not GOOGLE_API_KEY:
        raise ValueError("Выбран Google, но секрет GEMINI_API_KEY не задан в Colab (значок ключа слева).")
    client_gemini = genai.Client(api_key=GOOGLE_API_KEY.strip())
else:
    from openai import AsyncOpenAI
    DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
    if not DEEPSEEK_API_KEY:
        raise ValueError("Секрет DEEPSEEK_API_KEY не задан в Colab (значок ключа слева).")
    client_deepseek_async = AsyncOpenAI(api_key=DEEPSEEK_API_KEY.strip(), base_url="https://api.deepseek.com/v1", timeout=API_TIMEOUT)
print(f"🤖 Провайдер: {AI_PROVIDER.upper()} | разметка: {BATCH_MODEL} | разборы: {SUMMARY_MODEL}")

# --- ДИСК: подключается лениво (как в v4.2) ---
from google.colab import drive
_DRIVE_READY = {'ok': os.path.exists('/content/drive/MyDrive')}

def ensure_drive(reason=''):
    """Монтирует Диск по требованию. Возвращает True, если Диск доступен."""
    if DRIVE_MODE == "never":
        return False
    if not _DRIVE_READY['ok']:
        print(f"🔗 Подключаю Google Диск{': ' + reason if reason else ''}...")
        try:
            drive.mount('/content/drive', force_remount=False)
            _DRIVE_READY['ok'] = os.path.exists('/content/drive/MyDrive')
        except Exception as e:
            print(f"⚠️ Не удалось подключить Диск: {e}")
            _DRIVE_READY['ok'] = False
    if _DRIVE_READY['ok']:
        # ЧТО ИЗМЕНЕНО (v5.0): папки создаются и тогда, когда Диск был подключён раньше (второй прогон в сессии)
        # ПОЧЕМУ: иначе запись истории и кэша на Диск падала, если папки history ещё нет
        os.makedirs(TARGET_DIR, exist_ok=True)
        os.makedirs(HISTORY_DIR, exist_ok=True)
    return _DRIVE_READY['ok']

os.makedirs(CACHE_DIR, exist_ok=True)
if USE_FILE_PICKER:
    os.makedirs(TARGET_DIR, exist_ok=True)
if DRIVE_MODE == "always":
    ensure_drive('режим always')

# --- ЗАГРУЗКА ФАЙЛОВ ---
# Можно выбрать сразу несколько: выгрузку, template_v5.html, report_v5.py, history.json, markup_cache_v5.json
file_name, file_bytes = None, None
if USE_FILE_PICKER:
    print("📁 Выберите выгрузку (JSON) и при необходимости template_v5.html, report_v5.py, history.json, markup_cache_v5.json:")
    uploaded = files.upload()
    for _name, _bytes in uploaded.items():
        low = _name.lower()
        if low.endswith(('.html', '.py')):
            # «report_v5 (1).py» после скачивания браузером — сохраняется как report_v5.py; мажор всё равно проверяется ниже
            _m = re.match(r"(report|template)_v(\d+)", low)
            _target = f"{_m.group(1)}_v{_m.group(2)}{os.path.splitext(low)[1]}" if _m else _name
            with open(os.path.join(CACHE_DIR, _target), 'wb') as _f:
                _f.write(_bytes)
            print(f"🎨 Принят в кэш: {_name}" + (f" → {_target}" if _target != _name else ""))
        elif 'markup_cache' in low and low.endswith('.json'):
            with open(AI_CACHE_FILE, 'wb') as _f:
                _f.write(_bytes)
            print(f"💾 Кэш ИИ принят: {_name}")
        elif 'history' in low and low.endswith('.json'):
            with open(os.path.join(CACHE_DIR, 'history.json'), 'wb') as _f:
                _f.write(_bytes)
            print(f"🗓️ История недель принята: {_name}")
        elif low.endswith(('.json', '.csv')) and file_name is None:
            file_name, file_bytes = _name, _bytes
    if file_name is None:
        raise FileNotFoundError("Среди выбранных файлов нет выгрузки (.json или .csv).")
else:
    if not ensure_drive('нужна выгрузка'):
        raise FileNotFoundError("Без Диска выгрузку взять неоткуда: USE_FILE_PICKER = True или DRIVE_MODE != never.")
    SOURCE_DIR = next((p for p in SOURCE_DIR_VARIANTS if os.path.exists(p) and glob.glob(os.path.join(p, '*.*'))), None)
    if not SOURCE_DIR:
        raise FileNotFoundError("Исходные директории на Google Диске не найдены.")
    latest_file = max(glob.glob(os.path.join(SOURCE_DIR, '*.json')) + glob.glob(os.path.join(SOURCE_DIR, '*.csv')), key=os.path.getmtime)
    file_name = os.path.basename(latest_file)
    file_bytes = open(latest_file, 'rb').read()
print(f"📥 Выгрузка: {file_name}")


def find_file(name):
    """Файл по точному имени: кэш сессии, потом Диск (DRIVE_ROOT). Копия с Диска кладётся в кэш."""
    p = os.path.join(CACHE_DIR, name)
    if os.path.exists(p):
        return p
    if ensure_drive(f'нужен {name}'):
        for dirpath, _, fnames in os.walk(DRIVE_ROOT):
            for f in fnames:
                if f.lower() == name.lower():
                    src = os.path.join(dirpath, f)
                    try:
                        import shutil
                        shutil.copy(src, p)
                        return p
                    except Exception:
                        return src
    return None


# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО (v5.0): модуль и шаблон берутся по точному имени и проверяются по мажору; «самый свежий template*.html» больше не берётся
# ПОЧЕМУ: правило версий (CLAUDE.md, my-FT §9): несовместимую пару нельзя собрать молча; spec.md 7.5 — выбор шаблона без контроля мажора
# ----------------------------------------------------------------------
MODULE_PATH = find_file(MODULE_FILE)
if not MODULE_PATH:
    raise FileNotFoundError(f"Не найден {MODULE_FILE}: выберите его в окне загрузки вместе с выгрузкой или положите в {DRIVE_ROOT} на Диске.")
_spec = importlib.util.spec_from_file_location("report_v5", MODULE_PATH)
rv5 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rv5)
if rv5.VERSION.split('.')[0] != NOTEBOOK_VERSION.split('.')[0]:
    raise ValueError(f"{MODULE_FILE} версии {rv5.VERSION}, ноутбук {NOTEBOOK_VERSION}: мажоры должны совпадать")
TEMPLATE_PATH = find_file(TEMPLATE_FILE)
if not TEMPLATE_PATH:
    raise FileNotFoundError(f"Не найден {TEMPLATE_FILE}: выберите его в окне загрузки вместе с выгрузкой или положите в {DRIVE_ROOT} на Диске.")
TEMPLATE_TEXT = open(TEMPLATE_PATH, encoding='utf-8').read()
print(f"🧩 Модуль {MODULE_FILE} {rv5.VERSION}, шаблон {TEMPLATE_FILE} {rv5.check_template(TEMPLATE_TEXT)} — мажоры совпадают")

# --- ИСТОРИЯ НЕДЕЛЬ: кэш сессии + Диск ---
def load_history():
    recs = []
    p = os.path.join(CACHE_DIR, 'history.json')
    if os.path.exists(p):
        try:
            recs += json.load(open(p, encoding='utf-8'))
        except Exception as e:
            print(f"⚠️ Кэш истории повреждён, игнорирую: {e}")
    if DRIVE_MODE == "always" and ensure_drive('история недель'):
        for hf in glob.glob(os.path.join(HISTORY_DIR, 'week_*.json')):
            try:
                recs.append(json.load(open(hf, encoding='utf-8')))
            except Exception as e:
                print(f"⚠️ Пропущен повреждённый файл истории {os.path.basename(hf)}: {e}")
    return rv5.normalize_history(recs)

HISTORY = load_history()
print(f"🗓️ История: недель {len(HISTORY)}" + (f", последняя W{HISTORY[-1]['week_num']} {HISTORY[-1]['year']}" if HISTORY else ""))

# --- ВЫГРУЗКА → ОТЧЁТНАЯ НЕДЕЛЯ ---
if file_name.lower().endswith('.json'):
    RAW = json.loads(file_bytes.decode('utf-8-sig'))
else:
    RAW = list(csv.DictReader(io.StringIO(file_bytes.decode('utf-8-sig')), delimiter=';'))
R = rv5.load(RAW, file_name, REPORT_WEEK or None, HISTORY, demo='_demo_tab' in file_name)   # демо-выгрузка: блоки на выдуманных табличных частях помечаются
print(f"📅 Отчёт: W{R.WEEK} · {R.PER} · срез {R.NOW:%d.%m.%Y %H:%M} (Т-65: возраст открытых — от даты выгрузки)")

🤖 Провайдер: DEEPSEEK | разметка: deepseek-chat | разборы: deepseek-reasoner
📁 Выберите выгрузку (JSON) и при необходимости template_v5.html, report_v5.py, history.json, markup_cache_v5.json:


Saving sppr_dump_20260929_180748_842rec — копия.json to sppr_dump_20260929_180748_842rec — копия (3).json
📥 Выгрузка: sppr_dump_20260929_180748_842rec — копия (3).json
🧩 Модуль report_v5.py 5.1, шаблон template_v5.html 5.0 — мажоры совпадают
🗓️ История: недель 9, последняя W39 2026
Загружено 842 строк; отброшено статусов старой доски: 3 ['00-00005871', '00-00005988', '00-00006138']
Период: W39 21.09–25.09.2026 (по максимальной дате решения 25.09 15:23); срез 29.09 18:07
Табличные части: {'tab_trud': 0, 'tab_status': 0}; отброшено строк {}; расхождений tab_trud с AmountTrud 0
Пришло 125, решено 133, отклонено 4, открыто 137
📅 Отчёт: W39 · 21.09–25.09.2026 · срез 29.09.2026 18:07 (Т-65: возраст открытых — от даты выгрузки)


# БЛОК 2: ВЫЗОВ ИИ — ОБЩИЙ ВХОД, КЭШ, ПРОВЕРКА, ЖУРНАЛ

In [5]:
# =====================================================================
# [ЯЧЕЙКА 2] ВЫЗОВ ИИ: ОДИН ВХОД ДЛЯ ВСЕХ ЭТАПОВ, КЭШ, ПРОВЕРКА ОТВЕТА, ЖУРНАЛ
# =====================================================================
# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО (v5.0): вызов провайдера — одна функция ai_call на все этапы (в v4 — 4 копии в ячейках 3, 4, 5, 5Б);
#               ответ — всегда JSON, его проверяет код (rv5.validate) и при ошибке переспрашивает модель;
#               ответы кэшируются; пауза перед повтором — вне слота параллельности; время и токены этапов — в журнал
# ПОЧЕМУ: HANDOFF, разд. 0: проверка номеров и ключей кодом с повтором; spec.md 7.6 — дублированный вызов провайдера;
#         ускорение без потери качества; Т-63 — повторный прогон не платит заново
# ----------------------------------------------------------------------
SEM = asyncio.Semaphore(MAX_CONCURRENCY)
RUN_LOG = []        # этапы прогона: этап, модель, вызовы, из кэша, токены, секунды — в паспорт отчёта и в лог


class Stage:
    """Строка журнала прогона для одного этапа ИИ."""
    def __init__(self, name, model):
        self.rec = dict(stage=name, model=model, calls=0, cached=0, tokens=0, tok_in=0, tok_out=0, sec=0.0, note="")
        self.t0 = time.time()
        RUN_LOG.append(self.rec)

    def add(self, tok):
        self.rec["calls"] += 1
        self.rec["tok_in"] += tok[0]; self.rec["tok_out"] += tok[1]; self.rec["tokens"] += tok[2]

    def done(self, note=""):
        self.rec["sec"] = round(time.time() - self.t0, 1)
        self.rec["note"] = note
        print(f"⏱️ {self.rec['stage']}: {self.rec['sec']:.0f} с, вызовов {self.rec['calls']}, из кэша {self.rec['cached']}, токенов {self.rec['tokens']}"
              + (f" — {note}" if note else ""))


# --- кэш ответов ИИ: разметка по id и ответы целых вызовов по хэшу промпта ---
def load_ai_cache():
    paths = [AI_CACHE_FILE] + ([os.path.join(HISTORY_DIR, os.path.basename(AI_CACHE_FILE))] if DRIVE_MODE == "always" and ensure_drive('кэш ИИ') else [])
    cache = {"tickets": {}, "open_old": {}, "calls": {}}
    for p in paths:
        if not os.path.exists(p):
            continue
        try:
            d = json.load(open(p, encoding='utf-8'))
            if isinstance(d, dict) and "tickets" in d:
                for k in cache:
                    cache[k].update(d.get(k) or {})
            else:
                print(f"ℹ️ {os.path.basename(p)} — кэш другой версии ноутбука, не используется")
        except Exception as e:
            print(f"⚠️ Кэш ИИ {p} не прочитан ({e}) — игнорирую")
    return cache


def save_ai_cache():
    # ответы целых вызовов старше 60 дней не храним — файл не растёт бесконечно
    AI_CACHE["calls"] = {k: v for k, v in AI_CACHE["calls"].items() if time.time() - v.get("ts", 0) < 60 * 86400}
    with open(AI_CACHE_FILE, 'w', encoding='utf-8') as f:
        json.dump(AI_CACHE, f, ensure_ascii=False)
    if DRIVE_MODE == "always" and ensure_drive('запись кэша ИИ'):
        with open(os.path.join(HISTORY_DIR, os.path.basename(AI_CACHE_FILE)), 'w', encoding='utf-8') as f:
            json.dump(AI_CACHE, f, ensure_ascii=False)


AI_CACHE = load_ai_cache()
print(f"💾 Кэш ИИ: обращений {len(AI_CACHE['tickets'])}, старых открытых {len(AI_CACHE['open_old'])}, вызовов {len(AI_CACHE['calls'])}"
      + ("" if REUSE_AI else " — REUSE_AI = False: ответы целых вызовов не переиспользуются"))
sha1 = lambda *parts: hashlib.sha1("\x1f".join(map(str, parts)).encode("utf-8")).hexdigest()


async def ai_call(model, system, user, json_mode=True, max_tokens=None):
    """Один вызов модели. Возвращает (текст, (токены входа, выхода, всего))."""
    reasoner = model == SUMMARY_MODEL
    if AI_PROVIDER == "google":
        cfg = {'system_instruction': system, 'temperature': 0.2 if reasoner else 0.1}
        if json_mode: cfg['response_mime_type'] = 'application/json'
        if max_tokens: cfg['max_output_tokens'] = max_tokens
        resp = await client_gemini.aio.models.generate_content(model=model, contents=user, config=cfg)
        u = resp.usage_metadata
        tok = ((u.prompt_token_count or 0), (u.candidates_token_count or 0), (u.total_token_count or 0)) if u else (0, 0, 0)
        text = resp.text
    else:
        kw = dict(model=model, messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
        if not reasoner:            # у модели с рассуждением JSON — по инструкции в промпте; temperature она не принимает
            kw['temperature'] = 0.1
            if json_mode: kw['response_format'] = {"type": "json_object"}
            if max_tokens: kw['max_tokens'] = max_tokens
        resp = await client_deepseek_async.chat.completions.create(**kw)
        if not getattr(resp, 'choices', None):
            raise ValueError("пустой ответ API (нет choices)")
        u = resp.usage
        tok = ((u.prompt_tokens or 0), (u.completion_tokens or 0), (u.total_tokens or 0)) if u else (0, 0, 0)
        text = resp.choices[0].message.content
    if not text or not text.strip():
        raise ValueError("пустой ответ модели")
    return text, tok


def parse_json(text):
    """JSON из ответа модели: без ```json, от первой { или [ до последней } или ]."""
    t = text.strip().replace('```json', '').replace('```', '').strip()
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        i = min([x for x in (t.find('{'), t.find('[')) if x >= 0], default=-1)
        j = max(t.rfind('}'), t.rfind(']'))
        if i < 0 or j <= i:
            raise
        return json.loads(t[i:j + 1])


async def ask_json(st, model, system, user, check=None, pick=None, tries=3, json_mode=True, max_tokens=None, use_cache=True):
    """Вызов с проверкой: ответ, не прошедший check, переспрашивается с перечнем ошибок; последняя попытка — годные строки.
    Годный ответ кэшируется по хэшу промпта. Возвращает объект ответа или None."""
    key = sha1(model, system, user)
    if use_cache and REUSE_AI and key in AI_CACHE["calls"]:
        st.rec["cached"] += 1
        return AI_CACHE["calls"][key]["v"]
    feedback, last, delay = "", None, SLEEP_TIME
    for attempt in range(1, tries + 1):
        if attempt > 1:
            await asyncio.sleep(delay); delay *= 2            # пауза — вне слота SEM
        try:
            async with SEM:
                text, tok = await ai_call(model, system, user + feedback, json_mode, max_tokens)
            st.add(tok)
            obj = parse_json(text)
            if pick: obj = pick(obj)
        except Exception as e:
            print(f"⚠️ {st.rec['stage']}: попытка {attempt}/{tries} — {type(e).__name__}: {str(e)[:200]}")
            continue
        errs = check(obj) if check else []
        if not errs:
            if use_cache:
                AI_CACHE["calls"][key] = {"v": obj, "ts": time.time()}
            return obj
        last = (obj, errs)
        print(f"↩️ {st.rec['stage']}: ответ не прошёл проверку (попытка {attempt}/{tries}): {rv5.errors_text(errs)[:400]}")
        feedback = ("\n\nПРЕДЫДУЩИЙ ОТВЕТ ОТКЛОНЁН ПРОВЕРКОЙ КОДА: " + rv5.errors_text(errs)[:3000]
                    + "\nИсправьте именно эти места и пришлите ответ целиком заново, строго JSON.")
    if last:
        print(f"⛔ {st.rec['stage']}: после {tries} попыток оставлены только строки, прошедшие проверку")
        return rv5.sanitize(*last)
    return None

💾 Кэш ИИ: обращений 0, старых открытых 0, вызовов 0


# БЛОК 3: РАЗМЕТКА РЕШЁННЫХ ЗА ПЕРИОД

In [6]:
# =====================================================================
# [ЯЧЕЙКА 3] РАЗМЕТКА РЕШЁННЫХ ЗА ПЕРИОД
# =====================================================================
# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО (v5.0): разметка — по схеме v5 (data.md, разд. 11.1): поля v4 под короткими ключами + новые —
#               тип по сути с «служебной карточкой» (SVC), риск повтора, объект и причина правки с обоснованием,
#               «мог сделать сам», критичность; в промпт добавлены срок и трудозатраты (критерий «время несоразмерно задаче»).
#               Кэш по id (Т-63) и «Н/Д» при сбое (Т-64) — как в v4.1; пачка при обрыве — пополам, как в v4.
# ПОЧЕМУ: слайды v5 (правки, повторы, качество, аналитики) строятся на этих полях
# ----------------------------------------------------------------------
SYSTEM_TICKETS = f"""ЗАДАЧА: РАЗМЕТКА РЕШЁННЫХ ОБРАЩЕНИЙ В ПОДДЕРЖКУ 1С.
Вы — ведущий системный аналитик и архитектор 1С. Проведите жёсткий, технически объективный аудит каждого обращения пакета:
что случилось, что сделали, хорошо ли сделали и вернётся ли проблема.
Оценивайте строго. Оценка по умолчанию — «Х»; она ПОВЫШАЕТСЯ только за доказанную превентивность и ПОНИЖАЕТСЯ при любом
из признаков ниже. Не завышайте оценки из вежливости.

ОТВЕТ — строго JSON: {{"results": [объект на каждое обращение пакета]}}. Ключи объекта — ровно как ниже.
Пишите плотно, без вводных оборотов и без пересказа текста обращения; соблюдайте лимиты длины.

1. id — номер обращения из входа, как есть.
2. g — тема, до 60 символов: отглагольное существительное + конкретный объект системы («Создание контрагента»,
   «Изменение подписанта договора», «Пересчёт лимита БТД», «Ошибка проведения документа Заказ»).
   Запрещены абстракции («Ошибка системы», «Проблема с 1С», «Консультация», «Вопрос по ДО»).
   Не пишите «в 1С», «в системе» — они плодят синонимы одной темы. Одинаковые темы формулируйте одинаково.
3. s — суть, до 90 символов: что не работало или что просили. Сразу факт, своими словами, без «пользователь сообщает».
4. st — тип по сути, один код: I — инцидент; D — правка данных; K — консультация; P — права доступа; DB — дубль;
   SVC — служебная карточка: не обращение пользователя (учёт рабочего времени, встречи, планёрки, внутреннее поручение).
   Тип определяйте ПО СУЩЕСТВУ, не по полю «Тип (как указан)». РЕШЕНИЕ ВАЖНЕЕ ФОРМУЛИРОВКИ: инициатор часто описывает
   собственное непонимание как поломку. Тип определяется тем, ЧТО РЕАЛЬНО ПРИШЛОСЬ СДЕЛАТЬ:
   * объяснение, как пользоваться, куда нажать, почему так задумано, указание на ошибку самого пользователя — K,
     даже если в описании «ошибка», «не работает», «сбой», «не могу»;
   * правка, заведение или удаление данных — D;
   * выдача, снятие или делегирование прав — P;
   * исправление кода, настройки, перезапуск, передача в разработку, установка релиза — I;
   * ссылка на ранее созданное обращение — DB.
   Если из решения не видно, что делали, — по описанию: перестало работать то, что работало, — I; просьба поправить
   данные при штатной работе системы — D; вопрос «как сделать» — K; права и учётные записи — P; повтор — DB.
5. ta — до 120 символов: пусто, если тип по сути совпал с указанным; иначе — на что опирались (решение или описание)
   и по какому признаку. Для SVC — почему это не обращение.
6. gr — оценка решения, одна буква кириллицей: «О» отлично, «Х» хорошо, «П» плохо; для SVC — «—».
   «П» — при ЛЮБОМ из признаков:
   * прямые ручные правки в базе без устранения причины;
   * причина не названа: «исправлено», «устранено», «восстановлено» без объяснения, что было не так;
   * перенаправлено без ответа или закрыто «дубликат / ошибка» без объяснений;
   * ответ не решает проблему пользователя, формальная отписка, ответ не на тот вопрос;
   * срок несоразмерен задаче (простая консультация шла несколько рабочих дней — смотрите «Срок»);
   * проблема повторяется у разных пользователей, а решается точечно, без заявки на доработку;
   * удаление ЭЦП без согласования пользователя;
   * текст решения пустой, из одного слова или из даты.
   «Х» — стандартное корректное решение: названа причина, описано действие, проблема закрыта. Удаление ЭЦП
   с согласованием пользователя — «Х». Темы «Закрытие отчётного периода», «Встречи», «Планёрка» — «Х».
   Решение «Ваше обращение передано в Сервис Деск. Запрос на обслуживание номер …» — норма: не наше обращение, «Х».
   «О» — ТОЛЬКО за доказанную превентивность, не чаще чем у каждого пятого: заведена задача на исправление с номером,
   устранена причина для всех, дана инструкция или автоматизация, исключающая повтор. Подробного ответа для «О» мало.
7. ga — аргумент оценки, до 250 символов: за что снижена или повышена.
8. rec — одно проверяемое действие против повтора, до 120 символов: доработка, настройка, право, инструкция в базу
   знаний, автоматическая проверка. Не «усилить контроль», не «повысить внимательность». Повтор невозможен — пусто.
9. rr — риск повтора: «в» — причина не устранена или проблема у многих; «с» — средний; «н» — причина устранена или разовая ситуация.
10. tone — тон пользователя: 0 деловой; 1 скрытое недовольство (ирония, «сколько можно ждать», «опять не работает»);
    2 явное давление (требования эскалации, капслок в претензиях, угрозы жалоб); 3 оскорбления.
    Обязательно 0: восклицательные знаки в приветствиях и благодарностях, эмодзи; аббревиатуры капслоком (ЗУП, ГПД);
    «СРОЧНО» и «КРИТИЧНО», если процесс действительно стоит.
11. ea — до 120 символов: цитата или признак, если tone > 0; иначе пусто.
12. cx — сложность решения 1–5: 1 — кнопка или штатная правка; 2 — правка с проверкой; 3 — анализ и исправление цепочки
    документов; 4 — доработка или исправление обмена; 5 — перерасчёт регистров или архитектурный сбой. Для SVC — null.
13. crit — true, если это инцидент и он критичный: остановлена работа подразделения, отгрузки, оплата или расчёт
    зарплаты; налоговый или юридический риск; затронуто больше 5 пользователей. Иначе false.
14. ss — true, если пользователь мог сделать это сам при наличии права или инструкции; иначе false.
15. Для правок данных — st = «D» или тип указан «Изменение данных в системе»:
    obj — что правят, до 50 символов, «Объект: что», объект — справочник или документ, одинаково для одинаковых
    объектов: «Заказ-наряд: даты», «Контрагент: создание», «Договор: вид», «Пользователь: роли»;
    why — почему правили, строго одно из: {'; '.join(rv5.WHY_ACT)}.
      нет прав у пользователя — пользователь сделал бы сам, но нет права, поддержка правит за него;
      ошибка пользователя — пользователь ввёл неверно и просит исправить;
      дефект системы — данные испортил механизм системы (обмен, расчёт, загрузка, интеграция);
      штатная правка — регламентная операция, которую по процессу делает поддержка;
      разрыв процесса — в процессе нет шага (кадровое изменение не дошло до 1С, документ оформляют после факта);
      пробел НСИ — в справочнике нет нужного элемента или реквизита, нет владельца;
      нет функционала — система этого не умеет, правят руками вместо доработки;
      ошибка ввода НСИ — неверно заведён элемент справочника;
    wa — до 150 символов: на что в тексте опирались, выбирая причину.
    Для остальных обращений obj, why, wa — пустые строки.
"""

WHY_LC = {k.lower(): k for k in rv5.WHY_ACT}          # причина правки без учёта регистра → как в справочнике («пробел НСИ»)
GR_MAP = {"О": "О", "O": "О", "ОТЛИЧНО": "О", "Х": "Х", "X": "Х", "ХОРОШО": "Х", "П": "П", "P": "П", "ПЛОХО": "П",
          "—": "—", "-": "—", "–": "—", "": None}


def norm_ticket(a):
    """Ответ модели → строка разметки v5 или None (тогда обращение уходит на повтор)."""
    if not isinstance(a, dict):
        return None
    s_ = lambda k, n: str(a.get(k) or "").strip()[:n]
    st = str(a.get("st") or "").strip().upper()
    if st not in rv5.ST2TYPE:
        return None
    gr = GR_MAP.get(str(a.get("gr") or "").strip().upper())
    if gr is None and st == "SVC":
        gr = "—"
    g, s = s_("g", 80), s_("s", 120)
    if gr is None or not g or not s:
        return None
    rr = (str(a.get("rr") or "с").strip().lower() + "с")[0]
    yes = lambda v: v is True or str(v).strip().lower() in ("true", "1", "да")
    out = {"id": str(a.get("id")).strip(), "g": g, "s": s, "st": st, "ta": s_("ta", 160), "gr": gr, "ga": s_("ga", 300),
           "rec": s_("rec", 160), "rr": rr if rr in "всн" else "с", "tone": min(3, max(0, rv5._num(a.get("tone")))), "ea": s_("ea", 160),
           "cx": (min(5, max(1, rv5._num(a.get("cx"), 1))) if st != "SVC" else None),
           "crit": yes(a.get("crit")), "ss": yes(a.get("ss"))}
    why = WHY_LC.get(str(a.get("why") or "").strip().lower(), "")
    if why or s_("obj", 70):
        out.update(obj=s_("obj", 70), why=why, wa=s_("wa", 200))
    return out


def ticket_row(r):
    return (f"ID: {r['id']} | Инициатор: {r['cli']} | Аналитик: {r['anl']}\n"
            f"Раздел: {r['sec']} | Тип (как указан): {r['type']}\n"
            f"Зарегистрировано: {r['reg_dt']:%d.%m.%Y %H:%M} | Решено: {r['done_dt']:%d.%m.%Y %H:%M} | Срок: {r['h']} раб. ч (8 ч в дне) | Трудозатраты: {r['trud']:g} ч\n"
            f"Описание: {r['desc'][:PROMPT_TEXT_LIMIT]}\n"
            f"Решение: {r['sol'][:PROMPT_TEXT_LIMIT]}\n"
            f"----------------------------------------")


def ticket_key(r):
    return sha1(ticket_row(r), SYSTEM_TICKETS, BATCH_MODEL)


async def mark_chunk(st, chunk, tag):
    """Одна пачка. Возвращает (строки ответа, ok). ok=False — пачку переразметить меньшим размером."""
    prompt = "Разметьте пакет обращений:\n" + "\n".join(ticket_row(r) for r in chunk)
    delay = SLEEP_TIME
    for attempt in range(1, 4):
        if attempt > 1:
            await asyncio.sleep(delay); delay *= 2          # пауза — вне слота
        try:
            async with SEM:
                t0 = time.time()
                text, tok = await ai_call(BATCH_MODEL, SYSTEM_TICKETS, prompt, True, MAX_OUTPUT_TOKENS)
            st.add(tok)
            data = parse_json(text)
            data = data.get("results", []) if isinstance(data, dict) else data
            print(f"✅ Пакет {tag}: строк {len(chunk)}, {time.time() - t0:.0f} с, токенов {tok[2]}")
            return (data if isinstance(data, list) else []), True
        except json.JSONDecodeError as e:
            print(f"✂️ Пакет {tag}: ответ обрезан или не JSON ({e.msg}) — пачку из {len(chunk)} строк размечаем заново меньшим размером")
            return [], False
        except Exception as e:
            print(f"⚠️ Пакет {tag} (попытка {attempt}/3): {type(e).__name__}: {str(e)[:200]}")
    return [], False


async def run_ticket_markup(rows):
    st = Stage("Разметка решённых", BATCH_MODEL)
    done, todo = {}, []
    for r in rows:
        hit = AI_CACHE["tickets"].get(r["id"])
        if hit and hit.get("key") == ticket_key(r):
            done[r["id"]] = hit["row"]; st.rec["cached"] += 1
        else:
            todo.append(r)
    print(f"💾 Разметка: из кэша {len(done)}, к модели {len(todo)} из {len(rows)}")
    size, pno = BATCH_SIZE, 0
    while todo and pno < 6:
        pno += 1
        chunks = [todo[i:i + size] for i in range(0, len(todo), size)]
        print(f"🚀 Проход {pno}: строк {len(todo)}, пачек {len(chunks)} по {size}, потоков {MAX_CONCURRENCY}")
        res = await asyncio.gather(*[mark_chunk(st, c, f"{pno}.{i}") for i, c in enumerate(chunks, 1)])
        retry = []
        for (data, ok), chunk in zip(res, chunks):
            ids = {r["id"] for r in chunk}
            got = {}
            for a in data if ok else []:
                row = norm_ticket(a)
                if row and row["id"] in ids and row["id"] not in got:
                    got[row["id"]] = row
            for r in chunk:
                if r["id"] in got:
                    done[r["id"]] = got[r["id"]]
                    AI_CACHE["tickets"][r["id"]] = {"key": ticket_key(r), "row": got[r["id"]]}
                else:
                    retry.append(r)
        if retry:
            print(f"↩️ Не вернулись или не прошли проверку {len(retry)} строк — на повтор")
        todo = retry
        if not todo or size == 1:
            break
        size = max(1, size // 2)
    if todo:
        print(f"⛔ Не размечено {len(todo)} обращений: {[r['id'] for r in todo]} — в отчёте «Н/Д» (Т-64), следующий прогон разметит их заново")
    st.done(f"размечено {len(done)} из {len(rows)}" + (f", Н/Д {len(todo)}" if todo else ""))
    return [dict(done[r["id"]]) for r in rows if r["id"] in done]


MARK_ROWS = R.sol[:TEST_SAMPLE_SIZE] if TEST_SAMPLE_SIZE > 0 else R.sol
AI_MARKUP = {"tickets": await run_ticket_markup(MARK_ROWS)}
save_ai_cache()

💾 Разметка: из кэша 0, к модели 133 из 133
🚀 Проход 1: строк 133, пачек 9 по 15, потоков 4
✅ Пакет 1.2: строк 15, 14 с, токенов 12279
✅ Пакет 1.1: строк 15, 15 с, токенов 13564
✅ Пакет 1.3: строк 15, 14 с, токенов 12963
✅ Пакет 1.4: строк 15, 15 с, токенов 13134
✅ Пакет 1.6: строк 15, 12 с, токенов 12102
✅ Пакет 1.8: строк 15, 11 с, токенов 11272
✅ Пакет 1.5: строк 15, 13 с, токенов 11231
✅ Пакет 1.7: строк 15, 13 с, токенов 12553
✅ Пакет 1.9: строк 13, 12 с, токенов 11916
⏱️ Разметка решённых: 39 с, вызовов 9, из кэша 0, токенов 111014 — размечено 133 из 133


# БЛОК 4: ГРУППЫ ТЕМ

In [7]:
# =====================================================================
# [ЯЧЕЙКА 4] ГРУППЫ ТЕМ (ТОП-7 ТОЧЕЧНЫХ ПРОБЛЕМ)
# =====================================================================
# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО (v5.0, data.md, разд. 11.2): группировка — по темам с примерами сути и разделом (в v4 — только формулировки);
#               ориентир «8–15 групп» снят: одиночная группа допустима, свалки и группы-модули запрещены;
#               почти без объединений (W39 в v4.2: 111 групп на 111 тем) — один повтор с замечанием;
#               ответ кэшируется: повторный прогон той же выгрузки даёт тот же топ-7; переименование плохих названий — из v4
# ПОЧЕМУ: Т-44 — топ-7 точечных проблем считается по группам; в v4.2 группировка не объединяла ничего
# ----------------------------------------------------------------------
SYSTEM_GROUPS = """ЗАДАЧА: ГРУППЫ ТЕМ РЕШЁННЫХ ОБРАЩЕНИЙ.
Вы — ведущий системный аналитик 1С. Ниже — темы решённых обращений за неделю: номер, тема, примеры сути, разделы.
Сведите в группы темы, которые описывают ОДНУ И ТУ ЖЕ проблему или операцию одного объекта системы, чтобы руководитель
увидел повторы: одна мера — минус N обращений.
Правила:
1. Объединяйте темы про один объект и одну проблему, даже если они сформулированы по-разному:
   «Создание пользователя», «Заведение учётной записи 1С» — одна группа.
2. Разные объекты или разные проблемы одного объекта — разные группы. Не объединяйте по модулю или разделу:
   группы «Договоры», «Закупки», «ЭДО», «Кадры и пропуска», «Прочее», «Разное» запрещены.
3. Тема без пары — группа из одной темы. Это нормально.
4. Название — 3–7 слов: ОБЪЕКТ + ЧТО С НИМ: «Состояние «Проект» не передаётся из ЕРП в ДО», «Даты заказ-нарядов под
   документы подрядчика», «Создание пользователя 1С», «Затёрт справочник «Контрагенты»». Для НСИ можно «НСИ: контрагенты».
   Без перечислений через запятую и союз «и» — если тянет написать «и», это две группы.
5. Каждый номер темы — ровно в одной группе.
Ответ — строго JSON: {"groups": [{"name": "…", "items": [1, 5, 12]}]}"""
BAD_NAME_MARKERS = ['прочее', 'прочие', 'разное', 'другое', 'общие вопросы', 'иные']


def is_bad_group_name(name):
    low = str(name).strip().lower()
    return (not low or any(m in low for m in BAD_NAME_MARKERS) or ' и ' in low or low.count(',') >= 1 or len(low.split()) <= 1)


def groups_check(topics):
    soft = {"asked": False}           # «почти ничего не объединено» — переспросить один раз, потом принять как есть

    def check(obj):
        if not isinstance(obj, dict) or not isinstance(obj.get("groups"), list):
            return [((), "нужен объект {\"groups\": [...]}")]
        errs, seen = [], set()
        for i, g in enumerate(obj["groups"]):
            if not isinstance(g, dict) or not str(g.get("name") or "").strip() or not isinstance(g.get("items"), list):
                errs.append((("groups", i), "у группы нужны name и items")); continue
            for x in g["items"]:
                try:
                    n = int(x)
                except (TypeError, ValueError):
                    n = 0
                if not 1 <= n <= len(topics):
                    errs.append((("groups", i), f"номер темы {x} вне списка 1…{len(topics)}"))
                seen.add(n)
        if not errs and not soft["asked"] and len(topics) >= 20 and len(obj["groups"]) >= 0.9 * len(topics):
            soft["asked"] = True
            errs.append(((), f"почти ничего не объединено: {len(obj['groups'])} групп на {len(topics)} тем — объедините темы одной проблемы"))
        return errs
    return check


async def group_topics(rows):
    st = Stage("Группы тем", BATCH_MODEL)
    for a in rows:
        a.setdefault("tc", a["g"])           # tc — тема из разметки; g — группа (после этой ячейки)
    topics = sorted({a["tc"] for a in rows})
    if len(topics) <= 3:
        st.done("тем мало — группы не нужны"); return
    by_t = rv5.by(rows, lambda a: a["tc"])
    lines = []
    for n, t in enumerate(topics, 1):
        ex = by_t[t]
        secs = ", ".join(sorted({R.BYID[a["id"]]["sec"] for a in ex if a["id"] in R.BYID}))
        lines.append(f"{n}. {t} — суть: " + "; ".join(f"«{a['s']}»" for a in ex[:3]) + f" ({secs}; обращений {len(ex)})")
    user = f"Сгруппируйте {len(topics)} тем:\n" + "\n".join(lines)
    res = await ask_json(st, BATCH_MODEL, SYSTEM_GROUPS, user, check=groups_check(topics), max_tokens=MAX_OUTPUT_TOKENS)
    mapping = {}
    for g in (res or {}).get("groups", []):
        if not isinstance(g, dict): continue
        for x in g.get("items", []):
            try:
                t = topics[int(x) - 1]
            except (TypeError, ValueError, IndexError):
                continue
            mapping.setdefault(t, str(g.get("name") or "").strip() or t)
    # второй проход — только для плохих названий (v4): свалки, перечисления, одно слово
    bad = sorted({n for n in mapping.values() if is_bad_group_name(n)})
    if bad:
        listing = "\n".join(f"{i}. {n} — темы: " + "; ".join(t for t, g in mapping.items() if g == n)[:400] for i, n in enumerate(bad, 1))
        fix = await ask_json(st, BATCH_MODEL, SYSTEM_GROUPS,
                             "Названия групп ниже нарушают правило 4. Переформулируйте каждое по схеме ОБЪЕКТ + ЧТО С НИМ по темам внутри.\n"
                             + listing + '\nОтвет — строго JSON: {"names": ["новое название 1", "новое название 2"]}',
                             check=lambda o: [] if isinstance(o, dict) and isinstance(o.get("names"), list) and len(o["names"]) == len(bad) else [((), f"нужно ровно {len(bad)} названий в names")])
        for old, new in zip(bad, (fix or {}).get("names", [])):
            if isinstance(new, str) and new.strip() and not is_bad_group_name(new):
                mapping = {t: (new.strip() if g == old else g) for t, g in mapping.items()}
    for a in rows:
        a["g"] = mapping.get(a["tc"], a["tc"])     # не попала ни в одну группу — сама себе группа
    n_g = len({a["g"] for a in rows})
    st.done(f"тем {len(topics)} → групп {n_g}" + ("" if res else "; группировка не удалась — группы = темы"))


await group_topics(AI_MARKUP["tickets"])
save_ai_cache()

⏱️ Группы тем: 11 с, вызовов 2, из кэша 0, токенов 11472 — тем 124 → групп 106


# БЛОК 5: ОТКРЫТЫЕ, ПЛАНШЕТЫ, СКРЫТЫЕ ПОВТОРЫ

In [8]:
# =====================================================================
# [ЯЧЕЙКА 5] ОТКРЫТЫЕ: СТАРШЕ 30 ДНЕЙ, КРИТИЧНЫЕ ИНЦИДЕНТЫ; ПЛАНШЕТЫ; СКРЫТЫЕ ПОВТОРЫ
# =====================================================================
# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО (v5.0): новые этапы разметки (data.md, разд. 11.3): действие по каждому открытому старше 30 дней,
#               критичность открытых инцидентов, причина по каждому обращению про планшеты, скрытые повторы с цитатой;
#               после них разметка передаётся модулю (attach_ai) — считаются все метрики отчёта
# ПОЧЕМУ: Т-20, Т-22 — как сократить очередь; Т-45 — критичные выписать отдельно (в v4 критичность была только у решённых,
#         spec.md, разд. 6); Т-41 — планшеты; Т-42 — эмоции показывают скрытые повторы
# ----------------------------------------------------------------------
SYSTEM_OPEN_OLD = f"""ЗАДАЧА: СТАРЫЕ ОТКРЫТЫЕ ОБРАЩЕНИЯ.
Вы — руководитель поддержки 1С и разбираете очередь. Обращения ниже открыты больше 30 дней. Для каждого предложите
ОДНО действие, которое уберёт его из очереди или сдвинет с места.
ОТВЕТ — строго JSON: {{"results": [{{"id": "…", "ag": "…", "s": "…", "a": "…"}}]}} — объект на каждое обращение пакета.
ag — группа действия, строго один код:
  close — закрыть по регламенту: статус «Уточнение», вопрос пользователю задан, ответа нет;
  empty — закрыть: пустая карточка — нет описания или нечего делать;
  check — проверить: вероятно, уже решено — по тексту видно, что сделано, но карточка не закрыта;
  dup — объединить дубли: та же проблема, что в другой открытой карточке, — одна доработка;
  esc — эскалировать: анализ без движения, нужен срок или решение руководителя;
  plan — решить с разработкой: делать или отклонить доработку;
  rel — довести до релиза: исправление сделано или в тестировании, ждёт установки и проверки.
s — суть, до 90 символов, своими словами.
a — конкретное действие, до 160 символов: что сделать и с кем. Для dup — номер карточки, с которой объединить;
  если карточку можно сразу закрыть как дубль другой — начните «Дубль 00-000…: закрыть».
  Номера других обращений — только из индекса открытых ниже, точно как там."""

SYSTEM_OPEN_CRIT = """ЗАДАЧА: КРИТИЧНЫЕ ОТКРЫТЫЕ ИНЦИДЕНТЫ.
Вы — руководитель поддержки 1С. Ниже — все открытые инциденты. Отметьте только критичные: остановлена оплата, отгрузка,
расчёт зарплаты или работа подразделения; налоговый или юридический риск; затронуто больше 5 пользователей.
Неудобства, единичные ошибки с обходным путём и пожелания — не критичные.
ОТВЕТ — строго JSON: {"open_crit": [{"id": "…", "why": "что под угрозой, до 120 символов"}]}. Нет критичных — пустой список."""

SYSTEM_PLANSHET = f"""ЗАДАЧА: ПРИЧИНЫ ОБРАЩЕНИЙ ПО ПЛАНШЕТАМ.
Идёт внедрение планшетов (мобильный инженер), руководство контролирует, что мешает. Для каждого обращения ниже выберите
одну причину строго из списка:
«Вход и учётная запись» — не войти, учётная запись, пароль, права в мобильном клиенте;
«Функции мобильного инженера» — не работает функция: дефекты, отборы, метки, фото, файлы, обновление данных;
«Зависания и запуск» — зависает, не запускается, долго грузится;
«Связь в ремзонах» — нет сигнала, теряется связь;
«Доработки и прочее» — пожелания, доработки, прочее;
«Массовый сбой ДД.ММ» — только если в один день несколько обращений от разных людей про один и тот же сбой; ДД.ММ — дата этого дня.
ОТВЕТ — строго JSON: {{"planshet": {{"00-00012345": "причина", …}}}} — для каждого обращения из списка."""

SYSTEM_SIGNALS = """ЗАДАЧА: СКРЫТЫЕ ПОВТОРЫ И ЭМОЦИИ.
Вы — руководитель качества поддержки 1С. Ниже — обращения периода, где пользователь пишет «опять», «снова», «уже писал»,
«вновь», «до сих пор», «по-прежнему», раздражён или торопит. Такие фразы показывают то, чего не видно в цифрах:
проблема не первая, исправление не помогло, сроки сорваны.
Выберите настоящие сигналы — не вежливые формулы и не «срочно» при реальной остановке работы; не больше 10, самые важные.
ОТВЕТ — строго JSON: {"signals": [{"id": "…", "q": "…", "k": "…", "a": "…"}]}
q — дословная цитата из описания или решения этого обращения, до 140 символов; можно сократить многоточием, слова не менять;
k — сигнал коротко: «повтор», «повтор после «исправления»», «регресс доработки», «раздражение», «давление», «простой»;
a — вывод, до 200 символов: к какому прошлому обращению относится повтор (номер — только из индекса ниже) и что сделать.
Сигналов нет — пустой список."""

short = rv5.short
line_idx = lambda r: f"{r['id']} | рег {r['reg_dt']:%d.%m.%Y} | {r['sec']} | {r['type']} | {r['st']} | {short(r['anl'])} | {r['desc'][:90].replace(chr(10), ' ')}"


def open_row(r):
    return (f"ID: {r['id']} | Возраст: {r['age']} дн. | Статус: {r['st']} (держит: {r['holder']}) | Раздел: {r['sec']} | Тип: {r['type']}\n"
            f"Аналитик: {r['anl']} | Инициатор: {r['cli']} | Трудозатраты: {r['trud']:g} ч | Зарегистрировано: {r['reg_dt']:%d.%m.%Y}\n"
            f"Описание: {r['desc'][:PROMPT_TEXT_LIMIT]}\nКомментарии: {r['sol'][:800]}\n----------------------------------------")


def norm_open(a, ids):
    if not isinstance(a, dict) or str(a.get("id", "")).strip() not in ids or a.get("ag") not in rv5.OPEN_ACTIONS:
        return None
    txt = str(a.get("a") or "").strip()[:220]
    if any(x not in R.RAWID for x in rv5.ID_RX.findall(txt)):
        return None                                     # номер не из выгрузки — на повтор
    return {"id": str(a["id"]).strip(), "ag": a["ag"], "s": str(a.get("s") or "").strip()[:120], "a": txt}


async def run_open_old(rows):
    st = Stage("Открытые старше 30 дней", BATCH_MODEL)
    index = "ИНДЕКС ВСЕХ ОТКРЫТЫХ (для поиска дублей):\n" + "\n".join(line_idx(r) for r in sorted(R.opn, key=lambda r: r["sec"]))
    key = lambda r: sha1(open_row(r), SYSTEM_OPEN_OLD, BATCH_MODEL)
    done, todo = {}, []
    for r in rows:
        hit = AI_CACHE["open_old"].get(r["id"])
        if hit and hit.get("key") == key(r):
            done[r["id"]] = hit["row"]; st.rec["cached"] += 1
        else:
            todo.append(r)
    for size in (BATCH_SIZE, 5, 1):
        if not todo: break
        chunks = [todo[i:i + size] for i in range(0, len(todo), size)]

        async def one(chunk):
            ids = {r["id"] for r in chunk}
            # строки проверяются по одной (norm_open); не вернувшиеся — в следующий круг пачкой меньше; кэш — по id, не по вызову
            res = await ask_json(st, BATCH_MODEL, SYSTEM_OPEN_OLD, "Обращения:\n" + "\n".join(open_row(r) for r in chunk) + "\n\n" + index,
                                 pick=lambda o: o.get("results", []) if isinstance(o, dict) else o, max_tokens=MAX_OUTPUT_TOKENS, use_cache=False)
            return [x for x in (norm_open(a, ids) for a in (res if isinstance(res, list) else [])) if x]
        for rows_ in await asyncio.gather(*[one(c) for c in chunks]):
            for x in rows_:
                done.setdefault(x["id"], x)
        for r in todo:
            if r["id"] in done:
                AI_CACHE["open_old"][r["id"]] = {"key": key(r), "row": done[r["id"]]}
        todo = [r for r in todo if r["id"] not in done]
    st.done(f"разобрано {len(done)} из {len(rows)}" + (f"; без действия {len(todo)}" if todo else ""))
    return [done[r["id"]] for r in rows if r["id"] in done]


async def run_open_crit():
    st = Stage("Критичные открытые", BATCH_MODEL)
    lst = sorted([r for r in R.opn if r["type"] == "Инцидент"], key=lambda r: r["reg_dt"])
    if not lst:
        st.done("открытых инцидентов нет"); return []
    user = "Открытые инциденты:\n" + "\n".join(
        f"ID: {r['id']} | {r['age']} дн. | {r['st']} | {r['sec']} | {short(r['anl'])}\nОписание: {r['desc'][:700]}\nКомментарии: {r['sol'][:300]}\n---" for r in lst)
    res = await ask_json(st, BATCH_MODEL, SYSTEM_OPEN_CRIT, user, pick=lambda o: o.get("open_crit", []) if isinstance(o, dict) else o,
                         check=lambda o: rv5.validate(R, "open_crit", o), max_tokens=MAX_OUTPUT_TOKENS)
    st.done(f"критичных {len(res or [])} из {len(lst)}")
    return res or []


async def run_planshet():
    st = Stage("Планшеты", BATCH_MODEL)
    if not R.pl:
        st.done("обращений нет"); return {}
    lst = sorted(R.pl, key=lambda r: r["reg_dt"])
    user = "Обращения по планшетам:\n" + "\n".join(
        f"ID: {r['id']} | рег {r['reg_dt']:%d.%m.%Y %H:%M} | {r['st']} | инициатор {r['cli']}\nОписание: {r['desc'][:600]}\nРешение: {r['sol'][:300]}\n---" for r in lst)
    res = await ask_json(st, BATCH_MODEL, SYSTEM_PLANSHET, user, pick=lambda o: o.get("planshet", {}) if isinstance(o, dict) else o,
                         check=lambda o: rv5.validate(R, "planshet", o), max_tokens=MAX_OUTPUT_TOKENS)
    st.done(f"причина у {len(res or {})} из {len(lst)}; остальным — по словам в тексте")
    return res or {}


SIGNAL_RX = re.compile(r"опять|снова|вновь|уже писал|уже обращал|повторно|ранее|до сих пор|по-прежнему|который раз|сколько можно|почему\?|!!", re.I)


async def run_signals():
    st = Stage("Скрытые повторы", SUMMARY_MODEL)
    tone = {a["id"] for a in AI_MARKUP["tickets"] if a.get("tone", 0) > 0}
    per = {r["id"]: r for r in R.inc + R.sol + R.rej}.values()
    cand = sorted([r for r in per if r["id"] in tone or SIGNAL_RX.search(r["desc"] + " " + r["sol"])],
                  key=lambda r: (r["id"] not in tone, r["reg_dt"]))[:40]
    if not cand:
        st.done("сигналов-кандидатов нет"); return []
    sm = {a["id"]: a["s"] for a in AI_MARKUP["tickets"]}
    index = "\n".join(f"{r['id']} | {r['reg_dt']:%d.%m} | {r['sec']} | {(sm.get(r['id']) or r['desc'])[:70].replace(chr(10), ' ')}"
                      for r in sorted([r for r in R.rows if r["reg_dt"] >= R.W4S or r["is_open"]], key=lambda r: r["reg_dt"]))
    user = ("Обращения периода с признаками:\n" + "\n".join(
        f"ID: {r['id']} | рег {r['reg_dt']:%d.%m.%Y} | {r['sec']} | {r['st']} | инициатор {r['cli']}\nОписание: {r['desc'][:900]}\nРешение: {r['sol'][:500]}\n---" for r in cand)
        + "\n\nИНДЕКС ОБРАЩЕНИЙ ЗА 4 НЕДЕЛИ И ОТКРЫТЫХ (для ссылок на прошлые):\n" + index)
    res = await ask_json(st, SUMMARY_MODEL, SYSTEM_SIGNALS, user, pick=lambda o: o.get("signals", []) if isinstance(o, dict) else o,
                         check=lambda o: rv5.validate(R, "signals", o))
    st.done(f"сигналов {len(res or [])} из {len(cand)} кандидатов")
    return res or []


OLD_ROWS = sorted(R.old, key=lambda r: -r["age"])
OLD_ROWS = OLD_ROWS[:TEST_SAMPLE_SIZE] if TEST_SAMPLE_SIZE > 0 else OLD_ROWS
AI_MARKUP["open_old"], AI_MARKUP["open_crit"], AI_MARKUP["planshet"], AI_MARKUP["signals"] = await asyncio.gather(
    run_open_old(OLD_ROWS), run_open_crit(), run_planshet(), run_signals())
save_ai_cache()
rv5.attach_ai(R, AI_MARKUP)

⏱️ Планшеты: 4 с, вызовов 1, из кэша 0, токенов 12341 — причина у 47 из 47; остальным — по словам в тексте
⏱️ Критичные открытые: 4 с, вызовов 1, из кэша 0, токенов 10493 — критичных 13 из 53
⏱️ Открытые старше 30 дней: 11 с, вызовов 4, из кэша 0, токенов 63379 — разобрано 60 из 60
⏱️ Скрытые повторы: 72 с, вызовов 1, из кэша 0, токенов 62233 — сигналов 10 из 40 кандидатов
Разметка ИИ: решённых размечено 133 из 133; без разметки (Н/Д): []
Метрик: 271


# БЛОК 6: РАЗБОРЫ СЛАЙДОВ ИИ

In [9]:
# =====================================================================
# [ЯЧЕЙКА 6] РАЗБОРЫ СЛАЙДОВ ИИ (s1…s11) — JSON, ПРОВЕРКА КОДОМ, ПОВТОР ПРИ ОШИБКЕ
# =====================================================================
# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО (v5.0): вместо 6 свободных HTML-текстов (Root-Cause и 5 комментариев v4) — разбор каждого слайда в JSON
#               по схеме «факт → вывод → действие → кому → эффект» + постоянные блоки (решения недели, первопричины,
#               «закрыто, но вернётся», что чинить первым, лучшие решения, что поправить аналитикам, случай недели).
#               Модель видит то же, что читатель слайда (текст из модуля), каталог метрик и тексты обращений со слайда.
#               Цифры — ключами {метрика}, номера — только из выгрузки: проверяет код, ошибка — повторный запрос.
#               Девять разборов идут параллельно, сводный разбор недели (s1) — после них, по их выводам.
# ПОЧЕМУ: Т-7, Т-32, Т-33, Т-43, Т-50, Т-52; HANDOFF, разд. 0; spec.md 7.3 — входы ИИ v4 не совпадали со слайдами,
#         цифры в текстах ИИ не проверялись
# ----------------------------------------------------------------------
SYSTEM_SLIDES = """ЗАДАЧА: РАЗБОР СЛАЙДА ЕЖЕНЕДЕЛЬНОГО ОТЧЁТА О КАЧЕСТВЕ ПОДДЕРЖКИ 1С.
Вы — руководитель службы поддержки 1С и аналитик качества. Читатель — руководитель: ему нужно принять решения —
кого поправить, где изменить процесс, что чинить первым. Метрика без действия не нужна; вывод, по которому нельзя
принять решение, — лишний.
Правила ответа:
1. Ответ — один JSON-объект по схеме слайда, без markdown и пояснений вокруг.
2. Цифры из каталога метрик — ТОЛЬКО ключом в фигурных скобках: «{med_l2} раб. ч», «{a_ivanov_mis} из {a_ivanov_sol}».
   Код подставит значение. Ключи — только из каталога, точно как там. Не переписывайте цифры каталога руками и не
   пересчитывайте их. Число обращений, которые вы сами перечисляете в ids («3 инцидента»), можно писать числом.
3. Номера обращений (вида 00-00012345) — только из данных ниже, точно как там. Утверждение о конкретных обращениях
   подтверждайте номерами в ids.
4. items — 2–5 строк «факт → вывод → действие → кому → эффект»: fact — факт с цифрами; conc — вывод; act — одно
   проверяемое действие; who — кому (роль: руководитель поддержки, руководитель разработки, владелец процесса,
   владелец НСИ…); eff — эффект: сколько обращений или часов в неделю снимет, иначе пустая строка; ids — номера (можно []).
5. summary — 1–2 предложения: главный вывод слайда, с цифрами-ключами.
6. Только факты из данных. Запрещены общие слова: «усилить контроль», «повысить внимательность», «оптимизировать процессы».
7. По-русски, коротко: строка — не больше 200 символов."""

BASE_SCHEMA = ('"summary": "…", "items": [{"fact": "…", "conc": "…", "act": "…", "who": "…", "eff": "…", "ids": ["00-00012345"]}]')
SLIDES = {
    "s11": ("Сроки решения и долгострой",
            "Разберите долгострой — решённые за период, шедшие дольше {long_h_thr} раб. ч. Где ждали — у пользователя, в анализе, "
            "у разработки (по тексту решений); какие консультации и правки шли неделями и почему; есть ли концентрация у аналитиков, "
            "в разделах, в типах; сколько в сроке работы. Каждая строка items — конкретная причина задержки с номерами и мерой "
            "(срок ответа, маршрут, эскалация).", ""),
    "s2": ("Очередь: у кого мяч и как её сократить",
           "Что сделать на этой неделе, чтобы очередь сократилась: сколько закрыть без разработки (регламент, пустые, дубли, "
           "«вероятно решено») — с номерами; кто держит много старых без часов («свалка»); что стоит у разработки и почему; "
           "что копится в молодой очереди и завтра станет хвостом.", ""),
    "s3": ("Инциденты: что сломалось и что вернётся",
           "Что сломалось и где концентрация по разделам; сколько дефектов скрыто под другими типами; какие открытые критичные "
           "требуют срока сегодня. Дополнительно: root — 3–6 общих первопричин (несколько обращений — одна причина; ids — все "
           "обращения группы); notsolved — до 5 решённых за период обращений любого типа, которые формально закрыты, но вернутся; "
           "fix_first — 3–6 проблем, по которым больше всего обращений всех типов, решённых и открытых, и одна доработка на каждую.",
           ', "root": [{"cause": "…", "ids": ["…"], "fix": "что закроет группу", "kind": "доработка | регламент | настройка | права | обучение"}], '
           '"notsolved": [{"id": "…", "done": "что сделали", "why": "почему вернётся", "risk": "высокий | средний | низкий"}], '
           '"fix_first": [{"name": "проблема", "ids": ["…"], "act": "одна доработка"}]'),
    "s4": ("Ручные правки данных и НСИ",
           "Ручные правки данных без НСИ: почему правим (причины с обоснованием), какие правки на самом деле дефекты, кто просит "
           "повторно, что пользователь мог бы сделать сам. Каждая строка items — управленческое действие (право, проверка при вводе, "
           "доработка, обучение конкретных людей) с эффектом в обращениях в неделю. Отдельной строкой — НСИ: что правят и почему это "
           "постоянный поток. Дополнительно nsi_nonedit — классификация КАЖДОГО обращения из списка «не-правки в НСИ» ниже.",
           ', "nsi_nonedit": [{"id": "…", "cls": "что это по сути", "note": "что с этим делать"}]'),
    "s5": ("Повторы: что придёт снова",
           "Топ-7 групп проблем — какая мера снимет поток и сколько обращений; кто приносит повторы и почему; что пишут "
           "пользователи о повторах; в каком разделе растёт поток. Каждая строка — с номерами обращений.", ""),
    "s7": ("Планшеты: как идёт внедрение",
           "Внедрение на контроле руководства: падает ли поток, главная причина, проблемные устройства и места, что висит "
           "открытым, кто решает (первая линия, Сервис Деск). Строки items — действия для руководителя внедрения.", ""),
    "s6": ("Качество решений и классификации",
           "Все «Плохо» — что не так и что переделать; где и у кого путают тип и чем это опасно (дефекты не попадают в разработку); "
           "эмоциональный фон; служебные карточки в потоке. Дополнительно best — 3–5 лучших решений периода (оценка «О» или «Х»), "
           "которые надо тиражировать: почему эталон и что внести в базу знаний.",
           ', "best": [{"id": "…", "why": "почему эталон", "kb": "что внести в базу знаний"}]'),
    "s9": ("Аналитики: скорость, качество, что поправить",
           "Скорость к медиане типа, доля правок, дисциплина часов, «Плохо» и «Отлично», неверный тип, вторая линия на работе первой. "
           "Общий рейтинг не стройте — называйте конкретную ошибку конкретного человека с номером обращения. Дополнительно: fix — "
           "для каждого аналитика с конкретной проблемой одна строка «что поправить» с номером обращения (ключ — ФИО полностью из "
           "списка ниже); case_best и case_worst — лучший и худший случай недели среди решённых за период.",
           ', "fix": {"ФИО полностью": "что поправить, с номером обращения"}, "case_best": {"id": "…", "why": "…"}, "case_worst": {"id": "…", "why": "…"}'),
    "s8": ("Цена поддержки в часах",
           "Куда уходит время (разделы, типы, дорогие обращения), сколько ожидания на час работы, у кого часы не списаны, "
           "что вложено в открытые и «зомби». Строки items — меры с эффектом в часах.", ""),
    "s1": ("Неделя: поток, скорость, решения",
           "Сводный разбор недели — с него руководитель начинает отчёт. Ниже — выводы разборов всех остальных слайдов. "
           "summary — главный вывод недели: поток, скорость, очередь. items 2–4 — по цифрам первого слайда (поток, скорость линий, "
           "долгострой, служебные часы). decisions — ровно 5 решений руководителя недели: самые важные проблемы со всех слайдов "
           "(критичные инциденты, очередь, ручные правки — без НСИ, повторы, планшеты…) по схеме проблема — факт — действие — кому — "
           "эффект. root_all — 3–6 первопричин, которые объединяют обращения разных типов (правки, консультации, инциденты).",
           ', "decisions": [{"problem": "…", "fact": "…", "ids": ["…"], "act": "…", "who": "…", "eff": "…"}], '
           '"root_all": [{"cause": "…", "ids": ["…"], "act": "…", "who": "…"}]'),
}
CATALOG = rv5.catalog_text(R)
TEXTS = rv5.slide_texts(R)


def evidence(text, limit=60, budget=45000):
    """Описание и решение обращений, упомянутых на слайде: модель проверяет выводы по первоисточнику."""
    out, n = [], 0
    for i in dict.fromkeys(rv5.ID_RX.findall(text)):
        r = R.BYID.get(i) or R.RAWID.get(i)
        if not r: continue
        t = (f"{i} | {r.get('sec', '')} | {r.get('type', '')} | {r.get('st', '')} | {short(r.get('anl', ''))}\n"
             f"Описание: {str(r.get('desc', ''))[:400]}\nРешение: {str(r.get('sol', ''))[:500]}")
        if n + len(t) > budget or len(out) >= limit: break
        out.append(t); n += len(t)
    return "\n---\n".join(out) or "нет"


def slide_prompt(key, extra=""):
    title, task, schema = SLIDES[key]
    return (f"Слайд «{title}» (ключ {key}); отчёт W{R.WEEK}, период {R.PER}, срез открытых {R.NOW:%d.%m.%Y %H:%M}.\n"
            f"ЧТО НУЖНО: {task}\n"
            f"СХЕМА ОТВЕТА (JSON): {{{BASE_SCHEMA}{schema}}}\n\n"
            f"КАТАЛОГ МЕТРИК (ключ = значение — что это):\n{CATALOG}\n\n"
            f"ЧТО ВИДИТ ЧИТАТЕЛЬ НА СЛАЙДЕ:\n{TEXTS[key]}\n\n{extra}"
            f"ТЕКСТЫ ОБРАЩЕНИЙ СО СЛАЙДА (описание и решение, обрезаны):\n{evidence(TEXTS[key] + extra)}")


def slide_extra(key):
    if key == "s4":
        ne = [r for r in R.rows if r["sec"] == rv5.NSI_SECTION and r["type"] != "Изменение данных в системе"
              and (r["reg_dt"] >= R.W4S or r["is_open"] or (r["done_dt"] and r["done_dt"] >= R.W4S))]
        return ("НЕ-ПРАВКИ В НСИ ЗА 4 НЕДЕЛИ И ОТКРЫТЫЕ — классифицируйте каждое в nsi_nonedit:\n"
                + ("\n".join(f"{r['id']} | {r['type']} | {r['st']} | рег {r['reg_dt']:%d.%m.%Y} | {r['desc'][:200].replace(chr(10), ' ')}" for r in ne) or "нет — nsi_nonedit: []") + "\n\n")
    if key == "s9":
        return "АНАЛИТИКИ (ключ для fix — ФИО полностью, точно как здесь): " + "; ".join(R.anl_slug) + "\n\n"
    return ""


async def do_slide(key, extra=""):
    st = Stage(f"Разбор {key}", SUMMARY_MODEL)
    res = await ask_json(st, SUMMARY_MODEL, SYSTEM_SLIDES, slide_prompt(key, extra), check=lambda o: rv5.validate(R, "slide", o, key))
    st.done("готово" if res else "нет разбора — в отчёте нейтральная строка")
    return key, res


def slide_digest(key, s):
    """Разбор слайда простым текстом (цифры подставлены) — вход сводного разбора недели."""
    f = lambda t: rv5.fmt_plain(R, t)
    lines = [f"### {SLIDES[key][0]}", "Вывод: " + f(s.get("summary"))]
    lines += [f"- {f(x.get('fact'))} → {f(x.get('conc'))} → {f(x.get('act'))} ({x.get('who', '')}; {f(x.get('eff'))}) {' '.join(x.get('ids', []))}" for x in s.get("items", [])]
    for name, fields in rv5.SLIDE_EXTRA.get(key, {}).items():
        for x in s.get(name, []):
            lines.append(f"- [{name}] " + " | ".join(" ".join(x[k]) if k == "ids" else f(x.get(k)) for k in fields if k in x))
    for c in ("case_best", "case_worst"):
        if isinstance(s.get(c), dict): lines.append(f"- [{c}] {s[c].get('id')} — {s[c].get('why')}")
    return "\n".join(lines)


# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО (v5.1): модель больше не ставит статусы решениям прошлой недели (prev) — блок контроля скрыт до Т-70
# ПОЧЕМУ: my-FT В-21 = б: это контроль идей ИИ, а не поручений (замечание № 3); решения недели по-прежнему идут в историю (ячейка 7)
# ----------------------------------------------------------------------
others = await asyncio.gather(*[do_slide(k, slide_extra(k)) for k in rv5.AI_SLIDES if k != "s1"])
SLIDES_AI = {k: v for k, v in others if v}
digest = "\n\n".join(slide_digest(k, SLIDES_AI[k]) for k in rv5.AI_SLIDES if k in SLIDES_AI)
_, s1 = await do_slide("s1", f"ВЫВОДЫ РАЗБОРОВ ОСТАЛЬНЫХ СЛАЙДОВ:\n{digest}\n\n")
if s1:
    SLIDES_AI["s1"] = s1
AI_MARKUP["slides"] = SLIDES_AI
R.AI["slides"] = SLIDES_AI
save_ai_cache()
print(f"✦ Разборов слайдов: {len(SLIDES_AI)} из {len(rv5.AI_SLIDES)}" + ("" if len(SLIDES_AI) == len(rv5.AI_SLIDES) else
      " — у остальных в отчёте нейтральная строка, подробности выше"))

⏱️ Разбор s2: 32 с, вызовов 1, из кэша 0, токенов 31593 — готово
⏱️ Разбор s11: 35 с, вызовов 1, из кэша 0, токенов 20431 — готово
⏱️ Разбор s4: 41 с, вызовов 1, из кэша 0, токенов 38683 — готово
⏱️ Разбор s3: 50 с, вызовов 1, из кэша 0, токенов 30037 — готово
↩️ Разбор s5: ответ не прошёл проверку (попытка 1/3): items[1].fact: неизвестный ключ метрики {pl_n}
⏱️ Разбор s7: 76 с, вызовов 1, из кэша 0, токенов 19841 — готово
⏱️ Разбор s6: 80 с, вызовов 1, из кэша 0, токенов 29852 — готово
⏱️ Разбор s9: 102 с, вызовов 1, из кэша 0, токенов 38085 — готово
⏱️ Разбор s5: 104 с, вызовов 2, из кэша 0, токенов 66406 — готово
⏱️ Разбор s8: 107 с, вызовов 1, из кэша 0, токенов 21142 — готово
⏱️ Разбор s1: 36 с, вызовов 1, из кэша 0, токенов 41514 — готово
✦ Разборов слайдов: 10 из 10


# БЛОК 7: ИСТОРИЯ НЕДЕЛЬ, ОТЧЁТ, СОХРАНЕНИЕ

In [10]:
# =====================================================================
# [ЯЧЕЙКА 7] ИСТОРИЯ НЕДЕЛЬ v2, ВЁРСТКА ОТЧЁТА, СОХРАНЕНИЕ
# =====================================================================
# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО (v5.0): история v2 (data.md, разд. 6): за каждую неделю, которую выгрузка покрывает целиком, — поток
#               по типам и месяцам (пришло / решено / отклонено), очередь на срез и решения руководителя недели (Т-52, В-15);
#               отчёт верстает модуль report_v5 в оболочке template_v5.html; журнал этапов ИИ — в паспорт данных
# ПОЧЕМУ: Т-18 — месяцы копятся из истории; Т-52 — контроль решений в следующем отчёте; Т-51 — расход ИИ в паспорт
# ----------------------------------------------------------------------
DECISIONS = [{k: (d.get(k, []) if k == "ids" else rv5.fmt_plain(R, d.get(k, ""))) for k in ("problem", "fact", "ids", "act", "who", "eff")}
             for d in (R.AI["slides"].get("s1") or {}).get("decisions", [])]
if TEST_SAMPLE_SIZE > 0:
    print("🧪 ТЕСТ-РЕЖИМ: история не пишется")
else:
    new_recs = rv5.history_records(R, DECISIONS)
    old_map = {(r["year"], r["week_num"]): r for r in HISTORY}
    HISTORY = rv5.merge_history(HISTORY, new_recs)
    changed = [r for r in HISTORY if old_map.get((r["year"], r["week_num"])) != r]
    with open(os.path.join(CACHE_DIR, 'history.json'), 'w', encoding='utf-8') as f:
        json.dump(HISTORY, f, ensure_ascii=False, indent=1)
    # Диск — только если недели новые или цифры изменились (повторный прогон той же выгрузки Диск не трогает)
    if changed and ensure_drive('запись истории недель'):
        for rec in changed:
            with open(os.path.join(HISTORY_DIR, f"week_{rec['year']}_W{rec['week_num']:02d}.json"), 'w', encoding='utf-8') as f:
                json.dump(rec, f, ensure_ascii=False, indent=1)
    print(f"🗓️ История: записано недель {len(new_recs)} (изменилось {len(changed)}), решений недели {len(DECISIONS)}"
          + ("" if _DRIVE_READY['ok'] else " — Диск не подключён, history.json скачается ниже"))

html_output = rv5.render(R, TEMPLATE_TEXT, run_log=RUN_LOG)
base = rv5.report_file_name(R)
out_path = os.path.join(TARGET_DIR, f"{base}.html")
k = 1
while os.path.exists(out_path):
    out_path = os.path.join(TARGET_DIR, f"{base}_v{k}.html"); k += 1
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(html_output)
save_ai_cache()
print(f"✅ Отчёт сохранён: {out_path} ({len(html_output) // 1024} КБ)")
tok_total = sum(x["tokens"] for x in RUN_LOG)
print("📒 Журнал ИИ: " + "; ".join(f"{x['stage']} {x['sec']:.0f} с / {x['tokens']} ток." + (f" (кэш {x['cached']})" if x['cached'] else "") for x in RUN_LOG)
      + f" | всего токенов {tok_total}")
files.download(out_path)
if not _DRIVE_READY['ok']:
    print("💾 Диск не подключался — скачиваю history.json и кэш ИИ: приложите их к следующему запуску")
    for p in (os.path.join(CACHE_DIR, 'history.json'), AI_CACHE_FILE):
        if os.path.exists(p):
            files.download(p)

🗓️ История: записано недель 4 (изменилось 4), решений недели 5
✅ Отчёт сохранён: /content/result/Отчет по качеству W39 (21.09.2026).html (341 КБ)
📒 Журнал ИИ: Разметка решённых 39 с / 111014 ток.; Группы тем 11 с / 11472 ток.; Открытые старше 30 дней 11 с / 63379 ток.; Критичные открытые 4 с / 10493 ток.; Планшеты 4 с / 12341 ток.; Скрытые повторы 72 с / 62233 ток.; Разбор s11 35 с / 20431 ток.; Разбор s2 32 с / 31593 ток.; Разбор s3 50 с / 30037 ток.; Разбор s4 41 с / 38683 ток.; Разбор s5 104 с / 66406 ток.; Разбор s7 76 с / 19841 ток.; Разбор s6 80 с / 29852 ток.; Разбор s9 102 с / 38085 ток.; Разбор s8 107 с / 21142 ток.; Разбор s1 36 с / 41514 ток. | всего токенов 608516


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# БЛОК 8: ЛОГ В GOOGLE SHEETS (IS_DEBUG)

In [11]:
# =====================================================================
# [ЯЧЕЙКА 8] ЛОГ ЗАПУСКА В GOOGLE SHEETS (LOGS & TOKEN) — ПО ФЛАГУ IS_DEBUG
# =====================================================================
# ----------------------------------------------------------------------
# ЧТО ИЗМЕНЕНО (v5.0): ячейка перенесена из v4.2 после вёрстки — сбой записи лога не мешает отчёту; токены берутся
#               из журнала этапов: Batch — разметка решённых, Summary — разборы слайдов, Group — группы тем (модель
#               разметки), Incidents — открытые, планшеты, скрытые повторы. Смысл флага IS_DEBUG не менялся (HANDOFF: вне v5)
# ПОЧЕМУ: в v4 Group_Model писал модель разборов и Group_Total_Tokens всегда 0 (spec.md 7.5) — в новом коде не повторяем
# ----------------------------------------------------------------------
IS_DEBUG = 0 #@param {type:"integer"}

if IS_DEBUG == 1:
    import datetime as _dt
    import gspread
    from google.colab import auth
    from google.auth import default
    sh = None
    try:
        auth.authenticate_user()
        creds, _ = default()
        sh = gspread.authorize(creds).open('Logs')
    except Exception as e:
        print(f"⚠️ Таблица 'Logs' в Google Sheets не найдена или нет прав ({e}) — запись лога пропущена.")
    if sh is not None:
        run_id = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")
        try:
            ws_logs = sh.worksheet('Logs')
        except gspread.exceptions.WorksheetNotFound:
            ws_logs = sh.add_worksheet(title='Logs', rows=100, cols=10)
            ws_logs.append_row(['Run_ID', 'Timestamp', 'Year', 'Week', 'Total_Tickets', 'Status'])
        ws_logs.append_row([run_id, _dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), int(R.YEAR), int(R.WEEK), len(R.sol),
                            "TEST" if TEST_SAMPLE_SIZE > 0 else "SUCCESS"])
        TOKEN_HEADER = ['Run_ID', 'Batch_Model', 'Batch_Prompt_Tokens', 'Batch_Candidates_Tokens', 'Batch_Total_Tokens',
                        'Summary_Model', 'Summary_Prompt_Tokens', 'Summary_Candidates_Tokens', 'Summary_Total_Tokens',
                        'Group_Model', 'Group_Total_Tokens', 'Incidents_Total_Tokens']
        try:
            ws_tokens = sh.worksheet('Token')
        except gspread.exceptions.WorksheetNotFound:
            ws_tokens = sh.add_worksheet(title='Token', rows=100, cols=len(TOKEN_HEADER))
            ws_tokens.append_row(TOKEN_HEADER)
        pick = lambda pref: [x for x in RUN_LOG if x["stage"].startswith(pref)]
        tsum = lambda lst, k: sum(x[k] for x in lst)
        b, s_, g = pick("Разметка решённых"), pick("Разбор "), pick("Группы тем")
        inc = [x for x in RUN_LOG if x not in b + s_ + g]
        ws_tokens.append_row([run_id, BATCH_MODEL, tsum(b, "tok_in"), tsum(b, "tok_out"), tsum(b, "tokens"),
                              SUMMARY_MODEL, tsum(s_, "tok_in"), tsum(s_, "tok_out"), tsum(s_, "tokens"),
                              BATCH_MODEL, tsum(g, "tokens"), tsum(inc, "tokens")])
        print(f"📊 Лог и токены записаны в 'Logs' (Run ID: {run_id})")
else:
    print("⏭️ Запись в Google Sheets пропущена (IS_DEBUG != 1).")

📊 Лог и токены записаны в 'Logs' (Run ID: 20261002_000601)
